In [11]:
from computegraph import types as cgt
import computegraph as cg

import polars as pl
import numpy as npP

from jax import numpy as jnp, Array

from summer3.polarized.properties import Property, PropertyTable, LazyExpr
from summer3.polarized.categories import CategoryGroup, CategoryData
from summer3.polarized.flows import FlowSpec, source, dest
from summer3.polarized.expanding import scalar_to_expanding, catdata_to_expanding
from summer3.managed import ManagedArray, ManagedIndex

pl.Config.set_tbl_rows(64)
pl.Config.set_tbl_width_chars(None)

polars.config.Config

In [12]:
age = Property("age", ["infant", "child", "adult", "older"])
state = Property("state", ["S", "I", "R"])
severity = Property("severity", ["mild", "severe"])

pt = (
    PropertyTable.from_property(state)
    .stratify(severity, state == "I")
    .stratify(age, state)
)

In [ ]:
class Model:
    def __init__(self, base_strat):
        self.stratifications = []

In [13]:
class PTData:
    def __init__(self, data: Array, pt: PropertyTable):
        self.data = data
        self.pt = pt

    def filter(self, q) -> "PTData":
        idx_pt = self.pt.filter(q)
        return PTData(self.data[idx_pt.index.to_numpy()], idx_pt.reindex())

    def __repr__(self):
        return f"PropertData:\n{self.pt}\n{self.data}"

In [14]:
PTData(jnp.linspace(0.0, 1.0, len(pt)), pt).filter(age > "child")

PropertData:
PropertyTable
shape: (8, 4)
┌─────────┬────────────┬───────┬───────┐
│ state_0 ┆ severity_0 ┆ age_0 ┆ index │
│ ---     ┆ ---        ┆ ---   ┆ ---   │
│ str     ┆ str        ┆ str   ┆ i64   │
╞═════════╪════════════╪═══════╪═══════╡
│ S       ┆ null       ┆ adult ┆ 0     │
│ S       ┆ null       ┆ older ┆ 1     │
│ I       ┆ mild       ┆ adult ┆ 2     │
│ I       ┆ mild       ┆ older ┆ 3     │
│ I       ┆ severe     ┆ adult ┆ 4     │
│ I       ┆ severe     ┆ older ┆ 5     │
│ R       ┆ null       ┆ adult ┆ 6     │
│ R       ┆ null       ┆ older ┆ 7     │
└─────────┴────────────┴───────┴───────┘
[0.13333333 0.2        0.4        0.46666667 0.66666667 0.73333333
 0.93333333 1.        ]

In [15]:
from summer3.graph import Parameter, defer

In [16]:
class LazyPropData(cgt.LazyClass):
    def __init__(self, ref_node: cgt.GraphObject):
        self._ref = ref_node

    def filter(self, q) -> "LazyPropData":
        f = defer(PTData.filter)(self._ref, q)
        return LazyPropData(f)

    def evaluate(self, **sources):
        ref_res = self._ref.evaluate(**sources)
        return ref_res

In [17]:
Parameter("x", 1.0).evaluate(parameters={"x": 1.0})

1.0

In [8]:
def get_ptdata(x, pt):
    return PTData(jnp.linspace(0.0, x, len(pt)), pt)

In [10]:
ptdf = defer(get_ptdata)(Parameter("x", 2.0), pt)
gf = LazyPropData(ptdf)  # .filter(age > "infant")
gf.get_graph()

Traceback (most recent call last):
  File "_pydevd_bundle\\pydevd_cython.pyx", line 1609, in _pydevd_bundle.pydevd_cython.handle_exception
  File "/Users/s/dev/EMU/summer3proto/.pixi/envs/default/lib/python3.13/site-packages/debugpy/_vendored/pydevd/pydevd.py", line 2188, in do_wait_suspend
    keep_suspended = self._do_wait_suspend(thread, frame, event, arg, trace_suspend_type, from_this_thread, frames_tracker)
  File "/Users/s/dev/EMU/summer3proto/.pixi/envs/default/lib/python3.13/site-packages/debugpy/_vendored/pydevd/pydevd.py", line 2257, in _do_wait_suspend
    notify_event.wait(wait_timeout)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/Users/s/dev/EMU/summer3proto/.pixi/envs/default/lib/python3.13/threading.py", line 659, in wait
    signaled = self._cond.wait(timeout)
  File "/Users/s/dev/EMU/summer3proto/.pixi/envs/default/lib/python3.13/threading.py", line 363, in wait
    gotit = waiter.acquire(True, timeout)
KeyboardInterrupt


KeyError: '_var1'

In [ ]:
isinstance(gf, cgt.GraphObject)

True

In [ ]:
ptdata = PTData(jnp.linspace(0.0, 1.0, len(pt)), pt)